# PHẦN 2: PHÂN LOẠI THUỘC TÍNH & CHUYỂN ĐỔI KIỂU DỮ LIỆU (Feature Categorization)

**Dữ liệu:** House Prices – Advanced Regression Techniques (Kaggle)

**Mục tiêu:** phân tích độc lập từng thuộc tính để chuẩn hóa kiểu dữ liệu đúng với bản chất kỹ thuật:

1. Phân chia 79 biến giải thích thành nhóm **định lượng (Numeric)** và **định tính (Categorical)**.
2. Ép kiểu các biến số nguyên mang bản chất danh mục (`MSSubClass`, `MoSold`, `YrSold`) sang **Categorical**.
3. **Ordinal Encoding** cho các biến đánh giá chất lượng/điều kiện (`ExterQual`, `KitchenQual`, `BsmtQual`, `HeatingQC`, `GarageQual`, `PoolQC`, ...).

> Mỗi cell code bên dưới độc lập về mục đích, có thể copy riêng từng phần vào notebook của bạn. Cell 0 (import + đọc dữ liệu) là bắt buộc.

## 0. Import thư viện & đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

DATA_DIR = "data"   

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

TARGET = "SalePrice"
ID_COL = "Id"

print("Train:", train.shape, "| Test:", test.shape)

Train: (1460, 81) | Test: (1459, 80)


## 2.1. Phân chia nhóm biến: Numeric vs Categorical

Loại bỏ cột `Id` (định danh) và `SalePrice` (biến mục tiêu) → còn **79 biến giải thích**.
Phân nhóm dựa trên kiểu dữ liệu gốc mà pandas đọc được.

In [ ]:

feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET)]
print("Số biến giải thích:", len(feature_cols))

num_cols_raw = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]
cat_cols_raw = [c for c in feature_cols if c not in num_cols_raw]

print(f"Numeric     (gốc): {len(num_cols_raw)} biến")
print(f"Categorical (gốc): {len(cat_cols_raw)} biến")

print("\n--- Numeric ---")
print(num_cols_raw)
print("\n--- Categorical ---")
print(cat_cols_raw)

Số biến giải thích: 79
Numeric     (gốc): 36 biến
Categorical (gốc): 43 biến

--- Numeric ---
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

--- Categorical ---
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir

### Kiểm tra các biến số "đáng ngờ"
Các biến số có ít giá trị phân biệt (`nunique` nhỏ) cần xem xét kỹ: chúng là **số đo thực sự** hay **mã danh mục**?

In [ ]:
check = pd.DataFrame({
    "dtype":    train[num_cols_raw].dtypes.astype(str),
    "nunique":  train[num_cols_raw].nunique(),
    "min":      train[num_cols_raw].min(),
    "max":      train[num_cols_raw].max(),
    "missing":  train[num_cols_raw].isna().sum(),
}).sort_values("nunique")

check.head(20)   

,dtype,nunique,min,max,missing
BsmtHalfBath,int64,3,0.0,2.0,0
HalfBath,int64,3,0.0,2.0,0
Fireplaces,int64,4,0.0,3.0,0
KitchenAbvGr,int64,4,0.0,3.0,0
FullBath,int64,4,0.0,3.0,0
BsmtFullBath,int64,4,0.0,3.0,0
GarageCars,int64,5,0.0,4.0,0
YrSold,int64,5,2006.0,2010.0,0
PoolArea,int64,8,0.0,738.0,0
BedroomAbvGr,int64,8,0.0,8.0,0


In [ ]:

for c in ["MSSubClass", "MoSold", "YrSold"]:
    print(f"{c}: {sorted(train[c].unique().tolist())}")

MSSubClass: [20, 30, 40, 45, 50, 60, 70, 75, 80, 85, 90, 120, 160, 180, 190]
MoSold: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
YrSold: [2006, 2007, 2008, 2009, 2010]


## 2.2. Ép kiểu các biến số nguyên mang bản chất danh mục

| Biến | Lý do chuyển sang Categorical |
|---|---|
| `MSSubClass` | Là **mã loại hình nhà** (20, 30, 40, ...). Mã 60 không "gấp 3 lần" mã 20. |
| `MoSold` | **Tháng bán** (1–12): tính chu kỳ, tháng 12 và tháng 1 liền kề nhau, không có ý nghĩa cộng/trừ. |
| `YrSold` | **Năm bán** (2006–2010): chỉ có 5 mốc, dùng như nhãn thời kỳ (gắn với bối cảnh thị trường). |

> Giữ nguyên là Numeric: `OverallQual`, `OverallCond` (thang 1–10 đã là thứ tự số), `YearBuilt`, `YearRemodAdd`, `GarageYrBlt` (tuổi nhà có ý nghĩa đại số, có thể tạo biến `Age`).

In [ ]:

int_to_cat = ["MSSubClass", "MoSold", "YrSold"]

for df in (train, test):
    for c in int_to_cat:
      
        df[c] = df[c].astype(str).astype("category")

for c in int_to_cat:
    all_levels = sorted(set(train[c].cat.categories) | set(test[c].cat.categories))
    train[c] = train[c].cat.set_categories(all_levels)
    test[c]  = test[c].cat.set_categories(all_levels)

print(train[int_to_cat].dtypes)
print()
for c in int_to_cat:
    print(c, "->", list(train[c].cat.categories))

MSSubClass    category
MoSold        category
YrSold        category
dtype: object

MSSubClass -> ['120', '150', '160', '180', '190', '20', '30', '40', '45', '50', '60', '70', '75', '80', '85', '90']
MoSold -> ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
YrSold -> ['2006', '2007', '2008', '2009', '2010']


In [ ]:

num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

print(f"Sau ép kiểu -> Numeric: {len(num_cols)} | Categorical: {len(cat_cols)} | Tổng: {len(num_cols)+len(cat_cols)}")

Sau ép kiểu -> Numeric: 33 | Categorical: 46 | Tổng: 79


## 2.3. Mã hóa thang đo thứ tự (Ordinal Encoding)

Các biến đánh giá chất lượng/điều kiện có thứ tự rõ ràng từ yếu → xuất sắc:

`None = 0 < Po = 1 < Fa = 2 < TA = 3 < Gd = 4 < Ex = 5`

**Lưu ý quan trọng:** trong file CSV, giá trị `"NA"` (= *không có tiện ích*, ví dụ không có hầm/gara/hồ bơi) bị pandas đọc thành `NaN`.
Với các cột như `BsmtQual`, `GarageQual`, `PoolQC`, `FireplaceQu`... đây **không phải dữ liệu thiếu** mà là một mức hợp lệ → điền `"None"` rồi mã hóa thành **0**.
Ngược lại, với `ExterQual`, `KitchenQual`, `HeatingQC`... (ai cũng phải có) thì NaN (xuất hiện ở tập test) là **thiếu thật** → giữ NaN để xử lý ở phần Missing Values, không gán 0.

In [ ]:

qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

ordinal_core = ["ExterQual", "KitchenQual", "BsmtQual", "HeatingQC", "GarageQual", "PoolQC"]

ordinal_qual_extra = ["ExterCond", "BsmtCond", "GarageCond", "FireplaceQu"]

ordinal_qual_cols = ordinal_core + ordinal_qual_extra

for c in ordinal_qual_cols:
    print(f"{c:12s} | NaN: {train[c].isna().sum():4d} | giá trị: {sorted(train[c].dropna().unique())}")

ExterQual    | NaN:    0 | giá trị: ['Ex', 'Fa', 'Gd', 'TA']
KitchenQual  | NaN:    0 | giá trị: ['Ex', 'Fa', 'Gd', 'TA']
BsmtQual     | NaN:   37 | giá trị: ['Ex', 'Fa', 'Gd', 'TA']
HeatingQC    | NaN:    0 | giá trị: ['Ex', 'Fa', 'Gd', 'Po', 'TA']
GarageQual   | NaN:   81 | giá trị: ['Ex', 'Fa', 'Gd', 'Po', 'TA']
PoolQC       | NaN: 1453 | giá trị: ['Ex', 'Fa', 'Gd']
ExterCond    | NaN:    0 | giá trị: ['Ex', 'Fa', 'Gd', 'Po', 'TA']
BsmtCond     | NaN:   37 | giá trị: ['Fa', 'Gd', 'Po', 'TA']
GarageCond   | NaN:   81 | giá trị: ['Ex', 'Fa', 'Gd', 'Po', 'TA']
FireplaceQu  | NaN:  690 | giá trị: ['Ex', 'Fa', 'Gd', 'Po', 'TA']


In [ ]:

na_means_none = {"BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                 "GarageQual", "GarageCond", "GarageFinish", "FireplaceQu", "PoolQC", "Fence"}

def encode_ordinal(df, cols, mapping):
    """Map nhãn -> số thứ tự.
    - NaN ở cột thuộc na_means_none: điền 'None' (=0).
    - NaN ở cột khác: là thiếu dữ liệu thật -> giữ NaN (xử lý ở phần Missing Values).
    Báo lỗi nếu có nhãn lạ."""
    for c in cols:
        s = df[c].astype("object")
        if c in na_means_none:
            s = s.fillna("None")
        unknown = set(s.dropna().unique()) - set(mapping)
        assert not unknown, f"{c}: nhãn chưa có trong mapping: {unknown}"
        df[c] = s.map(mapping).astype("Int64")   # Int64 cho phép giữ NaN
    return df

for df in (train, test):
    encode_ordinal(df, ordinal_qual_cols, qual_map)

print(train[ordinal_qual_cols].dtypes)
train[ordinal_qual_cols].head()

ExterQual      Int64
KitchenQual    Int64
BsmtQual       Int64
HeatingQC      Int64
GarageQual     Int64
PoolQC         Int64
ExterCond      Int64
BsmtCond       Int64
GarageCond     Int64
FireplaceQu    Int64
dtype: object


,ExterQual,KitchenQual,BsmtQual,HeatingQC,GarageQual,PoolQC,ExterCond,BsmtCond,GarageCond,FireplaceQu
0,4,4,4,5,3,0,3,3,3,0
1,3,3,4,5,3,0,3,3,3,3
2,4,4,4,5,3,0,3,3,3,3
3,3,4,3,4,3,0,3,4,3,4
4,4,4,4,5,3,0,3,3,3,3


### (Tùy chọn) Mở rộng Ordinal Encoding cho các biến có thứ tự khác
Một số biến định tính khác cũng có thứ tự tự nhiên nhưng dùng thang riêng (theo `data_description.txt`).
Bỏ qua cell này nếu chỉ muốn mã hóa đúng các biến yêu cầu.

In [11]:
ordinal_custom = {
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Functional":   {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8},
    "LotShape":     {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
    "LandSlope":    {"Sev": 1, "Mod": 2, "Gtl": 3},
    "PavedDrive":   {"N": 0, "P": 1, "Y": 2},
    "CentralAir":   {"N": 0, "Y": 1},
    "Fence":        {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
}

for df in (train, test):
    for c, mp in ordinal_custom.items():
        encode_ordinal(df, [c], mp)

print(train[list(ordinal_custom)].dtypes)

BsmtExposure    Int64
BsmtFinType1    Int64
BsmtFinType2    Int64
GarageFinish    Int64
Functional      Int64
LotShape        Int64
LandSlope       Int64
PavedDrive      Int64
CentralAir      Int64
Fence           Int64
dtype: object


## 2.4. Tổng kết sau chuẩn hóa kiểu dữ liệu

In [12]:
ordinal_all = ordinal_qual_cols + [c for c in ordinal_custom if c in train.columns]

# Phân nhóm cuối cùng
cat_final = [c for c in feature_cols if isinstance(train[c].dtype, pd.CategoricalDtype)
             or not pd.api.types.is_numeric_dtype(train[c])]
ord_final = [c for c in ordinal_all if c in train.columns]
num_final = [c for c in feature_cols
             if pd.api.types.is_numeric_dtype(train[c]) and c not in ord_final]

summary = pd.DataFrame({
    "Nhóm": ["Numeric (liên tục/đếm)", "Ordinal (đã mã hóa số)", "Categorical (danh mục / nominal)", "TỔNG"],
    "Số biến": [len(num_final), len(ord_final), len(cat_final),
                len(num_final) + len(ord_final) + len(cat_final)],
})
print(summary.to_string(index=False))

print("\nNumeric    :", num_final)
print("\nOrdinal    :", ord_final)
print("\nCategorical:", cat_final)

                            Nhóm  Số biến
          Numeric (liên tục/đếm)       33
          Ordinal (đã mã hóa số)       20
Categorical (danh mục / nominal)       26
                            TỔNG       79

Numeric    : ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal']

Ordinal    : ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC', 'GarageQual', 'PoolQC', 'ExterCond', 'BsmtCond', 'GarageCond', 'FireplaceQu', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish', 'Functional', 'LotShape', 'LandSlope', 'PavedDrive', 'CentralAir', 'Fence']

Categorical: ['MSSubCl

In [ ]:

assert len(set(num_final) | set(ord_final) | set(cat_final)) == len(feature_cols) == 79
assert not (set(num_final) & set(cat_final)) and not (set(ord_final) & set(cat_final))

train.dtypes.value_counts()

int64       32
object      23
Int64       20
float64      3
category     1
category     1
category     1
Name: count, dtype: int64

In [ ]:

train.to_pickle("train_part2.pkl")
test.to_pickle("test_part2.pkl")

import json
json.dump({"numeric": num_final, "ordinal": ord_final, "categorical": cat_final},
          open("feature_groups.json", "w"), indent=2)
print("Đã lưu train_part2.pkl, test_part2.pkl, feature_groups.json")

Đã lưu train_part2.pkl, test_part2.pkl, feature_groups.json
